# Titanic Preprocessing and Modeling

This notebook downloads the Titanic competition data, performs quick exploratory data analysis, prepares the features with an sklearn preprocessing pipeline, and trains a simple baseline classifier.

## 1. Imports

In [ ]:
print("Hello")

In [ ]:
from pathlib import Path
import os

import kagglehub
from kagglehub.config import set_kaggle_credentials

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import skore
from dotenv import load_dotenv
load_dotenv()

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
print(os.getenv("KAGGLE_KEY"), os.getenv("KAGGLE_USERNAME"))

In [ ]:
# kagglehub.login()
set_kaggle_credentials(
    os.environ["KAGGLE_USERNAME"],
    os.environ["KAGGLE_KEY"],
)

## 2. Load the dataset

In [ ]:
data_dir = "/mnt/d/Programming/ITI/MLOps/Labs/mlops-lab0/data/raw"
competition_path = Path(kagglehub.competition_download("titanic", output_dir=data_dir, force_download=True))


def find_csv(root: Path, file_name: str) -> Path:
    matches = list(root.rglob(file_name))
    if not matches:
        raise FileNotFoundError(f"Could not find {file_name} under {root}")
    return matches[0]


train_path = find_csv(competition_path, "train.csv")
test_path = find_csv(competition_path, "test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Competition files stored in: {competition_path}")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

## 3. Quick EDA

In [ ]:
train_df.head()

In [ ]:
display(train_df.info())
display(train_df.describe(include="all").T)
display(train_df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

In [ ]:
train_df.nunique().sort_values()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=train_df, x="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Survival Distribution")

sns.countplot(data=train_df, x="Pclass", hue="Survived", ax=axes[0, 1])
axes[0, 1].set_title("Passenger Class vs Survival")

sns.countplot(data=train_df, x="Sex", hue="Survived", ax=axes[1, 0])
axes[1, 0].set_title("Sex vs Survival")

sns.histplot(data=train_df, x="Age", hue="Survived", kde=True, bins=30, ax=axes[1, 1])
axes[1, 1].set_title("Age Distribution by Survival")

plt.tight_layout()

In [ ]:
train_df.select_dtypes(include=["number"]).drop(["PassengerId","Survived"], axis=1).columns

In [ ]:
train_df.select_dtypes(include=["str"]).columns

In [ ]:
numeric_eda_cols = train_df.select_dtypes(include=["number"]).drop("PassengerId", axis=1).columns
corr = train_df[numeric_eda_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 4. Pre-processing with sklearn Pipeline

In [ ]:
train_df.columns

In [ ]:
dropped_cols = ["Cabin",
                "PassengerId",
                "Ticket",
                "Name",
                "Survived"]

target_column = "Survived"

X = train_df.drop(dropped_cols, axis=1).copy()
y = train_df[target_column].copy()
X_test_competition = test_df.drop(dropped_cols[:-1], axis=1).copy()

numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_onehot_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-1,
            ),
        ),
    ]
)

onehot_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_onehot_transformer, categorical_features),
    ]
)

hist_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_ordinal_transformer, categorical_features),
    ]
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_valid.shape

In [ ]:
transformed_sample = onehot_preprocessor.fit_transform(X_train)
print("Transformed feature matrix shape:", transformed_sample.shape)

## 5. Optuna Model Search

In [ ]:
list(
    range(len(numeric_features), len(numeric_features) + len(categorical_features))
)

In [ ]:
try:
    import optuna
    from catboost import CatBoostClassifier
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "Install optuna, xgboost, and catboost before running the optimization section."
    ) from exc

MODEL_NAMES = [
    "random_forest",
    "extra_trees",
    "gradient_boosting",
    "hist_gradient_boosting",
    "xgboost",
    "catboost",
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
hist_categorical_feature_idx = list(
    range(len(numeric_features), len(numeric_features) + len(categorical_features))
)


def prepare_catboost_frame(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = frame.copy()
    for column in categorical_features:
        prepared[column] = prepared[column].fillna("missing").astype(str)
    return prepared


X_train_catboost = prepare_catboost_frame(X_train)
X_valid_catboost = prepare_catboost_frame(X_valid)
X_test_catboost = prepare_catboost_frame(X_test_competition)


def sample_model_params(trial: optuna.Trial) -> dict:
    model_name = trial.suggest_categorical("model_name", MODEL_NAMES)
    params = {"model_name": model_name}

    if model_name == "random_forest":
        params.update(
            {
                "n_estimators": trial.suggest_int("rf_n_estimators", 200, 700),
                "max_depth": trial.suggest_int("rf_max_depth", 3, 16),
                "min_samples_split": trial.suggest_int("rf_min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 10),
                "max_features": trial.suggest_categorical(
                    "rf_max_features", ["sqrt", "log2", None]
                ),
            }
        )
    elif model_name == "extra_trees":
        params.update(
            {
                "n_estimators": trial.suggest_int("et_n_estimators", 200, 700),
                "max_depth": trial.suggest_int("et_max_depth", 3, 16),
                "min_samples_split": trial.suggest_int("et_min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("et_min_samples_leaf", 1, 10),
                "max_features": trial.suggest_categorical(
                    "et_max_features", ["sqrt", "log2", None]
                ),
            }
        )
    elif model_name == "gradient_boosting":
        params.update(
            {
                "n_estimators": trial.suggest_int("gb_n_estimators", 100, 500),
                "learning_rate": trial.suggest_float("gb_learning_rate", 0.01, 0.2, log=True),
                "max_depth": trial.suggest_int("gb_max_depth", 2, 6),
                "min_samples_split": trial.suggest_int("gb_min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("gb_min_samples_leaf", 1, 10),
                "subsample": trial.suggest_float("gb_subsample", 0.6, 1.0),
                "max_features": trial.suggest_categorical(
                    "gb_max_features", ["sqrt", "log2", None]
                ),
            }
        )
    elif model_name == "hist_gradient_boosting":
        params.update(
            {
                "learning_rate": trial.suggest_float("hgb_learning_rate", 0.01, 0.2, log=True),
                "max_iter": trial.suggest_int("hgb_max_iter", 150, 600),
                "max_leaf_nodes": trial.suggest_int("hgb_max_leaf_nodes", 15, 63),
                "max_depth": trial.suggest_int("hgb_max_depth", 3, 12),
                "min_samples_leaf": trial.suggest_int("hgb_min_samples_leaf", 10, 60),
                "l2_regularization": trial.suggest_float(
                    "hgb_l2_regularization", 1e-4, 10.0, log=True
                ),
                "max_bins": trial.suggest_int("hgb_max_bins", 64, 255),
            }
        )
    elif model_name == "xgboost":
        params.update(
            {
                "n_estimators": trial.suggest_int("xgb_n_estimators", 150, 700),
                "learning_rate": trial.suggest_float("xgb_learning_rate", 0.01, 0.2, log=True),
                "max_depth": trial.suggest_int("xgb_max_depth", 3, 10),
                "min_child_weight": trial.suggest_float(
                    "xgb_min_child_weight", 1.0, 10.0
                ),
                "subsample": trial.suggest_float("xgb_subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float(
                    "xgb_colsample_bytree", 0.6, 1.0
                ),
                "reg_alpha": trial.suggest_float("xgb_reg_alpha", 1e-4, 10.0, log=True),
                "reg_lambda": trial.suggest_float("xgb_reg_lambda", 1e-4, 10.0, log=True),
            }
        )
    else:
        params.update(
            {
                "iterations": trial.suggest_int("cat_iterations", 200, 800),
                "learning_rate": trial.suggest_float(
                    "cat_learning_rate", 0.01, 0.2, log=True
                ),
                "depth": trial.suggest_int("cat_depth", 4, 10),
                "l2_leaf_reg": trial.suggest_float("cat_l2_leaf_reg", 1.0, 10.0),
                "random_strength": trial.suggest_float(
                    "cat_random_strength", 1e-3, 10.0, log=True
                ),
                "bagging_temperature": trial.suggest_float(
                    "cat_bagging_temperature", 0.0, 5.0
                ),
                "border_count": trial.suggest_int("cat_border_count", 32, 255),
            }
        )

    return params


def build_model_from_params(params: dict):
    model_name = params["model_name"]

    if model_name == "random_forest":
        estimator = Pipeline(
            steps=[
                ("preprocessor", onehot_preprocessor),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=params["n_estimators"],
                        max_depth=params["max_depth"],
                        min_samples_split=params["min_samples_split"],
                        min_samples_leaf=params["min_samples_leaf"],
                        max_features=params["max_features"],
                        class_weight="balanced",
                        n_jobs=-1,
                        random_state=42,
                    ),
                ),
            ]
        )
        return estimator, X_train, X_valid, X_test_competition, {}

    if model_name == "extra_trees":
        estimator = Pipeline(
            steps=[
                ("preprocessor", onehot_preprocessor),
                (
                    "model",
                    ExtraTreesClassifier(
                        n_estimators=params["n_estimators"],
                        max_depth=params["max_depth"],
                        min_samples_split=params["min_samples_split"],
                        min_samples_leaf=params["min_samples_leaf"],
                        max_features=params["max_features"],
                        class_weight="balanced",
                        n_jobs=-1,
                        random_state=42,
                    ),
                ),
            ]
        )
        return estimator, X_train, X_valid, X_test_competition, {}

    if model_name == "gradient_boosting":
        estimator = Pipeline(
            steps=[
                ("preprocessor", onehot_preprocessor),
                (
                    "model",
                    GradientBoostingClassifier(
                        n_estimators=params["n_estimators"],
                        learning_rate=params["learning_rate"],
                        max_depth=params["max_depth"],
                        min_samples_split=params["min_samples_split"],
                        min_samples_leaf=params["min_samples_leaf"],
                        subsample=params["subsample"],
                        max_features=params["max_features"],
                        random_state=42,
                    ),
                ),
            ]
        )
        return estimator, X_train, X_valid, X_test_competition, {}

    if model_name == "hist_gradient_boosting":
        estimator = Pipeline(
            steps=[
                ("preprocessor", hist_preprocessor),
                (
                    "model",
                    HistGradientBoostingClassifier(
                        learning_rate=params["learning_rate"],
                        max_iter=params["max_iter"],
                        max_leaf_nodes=params["max_leaf_nodes"],
                        max_depth=params["max_depth"],
                        min_samples_leaf=params["min_samples_leaf"],
                        l2_regularization=params["l2_regularization"],
                        max_bins=params["max_bins"],
                        categorical_features=hist_categorical_feature_idx,
                        early_stopping=False,
                        random_state=42,
                    ),
                ),
            ]
        )
        return estimator, X_train, X_valid, X_test_competition, {}

    if model_name == "xgboost":
        estimator = Pipeline(
            steps=[
                ("preprocessor", onehot_preprocessor),
                (
                    "model",
                    XGBClassifier(
                        n_estimators=params["n_estimators"],
                        learning_rate=params["learning_rate"],
                        max_depth=params["max_depth"],
                        min_child_weight=params["min_child_weight"],
                        subsample=params["subsample"],
                        colsample_bytree=params["colsample_bytree"],
                        reg_alpha=params["reg_alpha"],
                        reg_lambda=params["reg_lambda"],
                        scale_pos_weight=scale_pos_weight,
                        tree_method="hist",
                        eval_metric="logloss",
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        )
        return estimator, X_train, X_valid, X_test_competition, {}

    estimator = CatBoostClassifier(
        iterations=params["iterations"],
        learning_rate=params["learning_rate"],
        depth=params["depth"],
        l2_leaf_reg=params["l2_leaf_reg"],
        random_strength=params["random_strength"],
        bagging_temperature=params["bagging_temperature"],
        border_count=params["border_count"],
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        verbose=0,
        allow_writing_files=False,
        random_state=42,
    )
    return estimator, X_train_catboost, X_valid_catboost, X_test_catboost, {
        "cat_features": categorical_features
    }


def objective(trial: optuna.Trial) -> float:
    params = sample_model_params(trial)
    estimator, search_train, _, _, fit_kwargs = build_model_from_params(params)

    fold_scores = []
    for fold_idx, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
        estimator_fold = clone(estimator)
        X_fold_train = search_train.iloc[train_idx]
        X_fold_valid = search_train.iloc[valid_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_valid = y_train.iloc[valid_idx]

        estimator_fold.fit(X_fold_train, y_fold_train, **fit_kwargs)
        y_valid_proba = estimator_fold.predict_proba(X_fold_valid)[:, 1]
        fold_score = roc_auc_score(y_fold_valid, y_valid_proba)
        fold_scores.append(fold_score)

        trial.report(float(np.mean(fold_scores)), step=fold_idx)

    return float(np.mean(fold_scores))

## 6. Run Optimization

In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(
    study_name="titanic_tree_search",
    direction="maximize",
    sampler=sampler,
)

for model_name in MODEL_NAMES:
    study.enqueue_trial({"model_name": model_name})

n_trials = 48
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print(f"Best ROC AUC: {study.best_value:.4f}")
print(f"Best model: {study.best_trial.params['model_name']}")
study.best_trial.params

In [ ]:
study.trials_dataframe(attrs=("number", "value", "state", "params"))\
    .sort_values("value", ascending=False)\
    .head(15)

## 7. Evaluate Best Model

In [ ]:
new_params = {}
best_params = study.best_trial.params.copy()
best_model_name = best_params["model_name"]
for key, value in best_params.items():
    if key.startswith("xgb_"):
        new_key = key[4:]
    else:
        new_key = key
    new_params[new_key] = value

In [ ]:
new_params.keys()

In [ ]:

best_model_pipeline, best_X_train, best_X_valid, X_test_final, best_fit_kwargs = build_model_from_params(new_params)

best_model_pipeline.fit(best_X_train, y_train, **best_fit_kwargs)
model_pipeline = best_model_pipeline

evaluation_report = skore.EstimatorReport(
    model_pipeline,
    fit=False,
    X_train=best_X_train,
    y_train=y_train,
    X_test=best_X_valid,
    y_test=y_valid,
    pos_label=1,
)

best_model_name

In [ ]:
evaluation_report

In [ ]:
display(
    evaluation_report.metrics.summarize(
        data_source="both",
        metric=["accuracy", "precision", "recall", "f1", "roc_auc", "log_loss", "brier_score"],
    ).frame(favorability=True)
)
evaluation_report.metrics.confusion_matrix().plot()

In [ ]:
competition_predictions = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": model_pipeline.predict(X_test_final),
    }
)

competition_predictions.head()